1. Data Collection
2. Data Cleaning
3. Database Creation
4. Analytics
5. KPIs
6. Charts
7. Power BI Outputs

In [1]:
import pandas as pd
import numpy as np
import random
import os
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()

# Create folders if missing
os.makedirs("NordicSpace_Project/data", exist_ok=True)

# ----------------------------
# SETTINGS
# ----------------------------
num_rows = 5000

cities = {
    "Helsinki": {"base_price": 150},
    "Espoo": {"base_price": 135},
    "Vantaa": {"base_price": 125},
    "Tampere": {"base_price": 115},
    "Turku": {"base_price": 110}
}

unit_sizes = {
    "Small Locker": 0.6,
    "5m² Unit": 1.0,
    "10m² Unit": 1.5,
    "Premium Climate": 1.8
}

channels = ["Website", "Walk-in", "Google Ads", "Referral", "Partner"]

# ----------------------------
# HELPER FUNCTIONS
# ----------------------------

def seasonal_multiplier(month):
    # Summer demand higher
    if month in [5,6,7,8]:
        return 1.25
    elif month in [11,12,1,2]:
        return 0.85
    return 1.0

def random_duration():
    return random.randint(1, 12)

# ----------------------------
# GENERATE BOOKINGS
# ----------------------------

rows = []

for i in range(1, num_rows + 1):
    
    city = random.choice(list(cities.keys()))
    unit = random.choice(list(unit_sizes.keys()))
    
    start_date = fake.date_between(start_date="-2y", end_date="today")
    duration = random_duration()
    end_date = start_date + timedelta(days=duration*30)
    
    month = start_date.month
    
    base = cities[city]["base_price"]
    unit_mult = unit_sizes[unit]
    season_mult = seasonal_multiplier(month)
    
    price = round(base * unit_mult * season_mult, 2)
    
    repeat = random.choice([0,1])
    
    rows.append({
        "booking_id": i,
        "customer_id": random.randint(1000,9999),
        "city": city,
        "unit_size": unit,
        "start_date": start_date,
        "end_date": end_date,
        "duration_months": duration,
        "monthly_price": price,
        "total_revenue": round(price * duration,2),
        "channel": random.choice(channels),
        "repeat_customer": repeat
    })

df = pd.DataFrame(rows)

# Save CSV
df.to_csv("NordicSpace_Project/data/bookings.csv", index=False)

print("bookings.csv created successfully")
print(df.head())

# ----------------------------
# GENERATE REVIEWS
# ----------------------------

positive_reviews = [
    "Excellent service and clean unit.",
    "Easy booking process and friendly staff.",
    "Very secure location and fair pricing.",
    "Perfect solution during my move.",
    "Climate unit protected my furniture well."
]

negative_reviews = [
    "Late payment fee was unclear.",
    "Difficult to find the location first time.",
    "Customer service response was slow.",
    "Needed better signage outside.",
    "Price felt slightly high."
]

ratings = [2,3,4,5]

review_lines = []

for _ in range(30):
    city = random.choice(list(cities.keys()))
    month = random.choice(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
    year = random.choice([2024,2025])
    
    if random.random() > 0.3:
        text = random.choice(positive_reviews)
        rating = random.choice([4,5])
    else:
        text = random.choice(negative_reviews)
        rating = random.choice([2,3])
    
    line = f'"{text}" - {city}, {month} {year}, {rating}/5'
    review_lines.append(line)

with open("NordicSpace_Project/data/reviews.txt", "w", encoding="utf-8") as f:
    for line in review_lines:
        f.write(line + "\n")

print("reviews.txt created successfully")

bookings.csv created successfully
   booking_id  customer_id     city        unit_size  start_date    end_date  \
0           1         7020   Vantaa         5m² Unit  2025-11-04  2026-08-01   
1           2         3127    Turku  Premium Climate  2025-04-21  2025-10-18   
2           3         2852  Tampere  Premium Climate  2025-01-06  2025-03-07   
3           4         1826  Tampere     Small Locker  2025-05-19  2025-06-18   
4           5         9860    Turku        10m² Unit  2024-10-23  2025-06-20   

   duration_months  monthly_price  total_revenue     channel  repeat_customer  
0                9         106.25         956.25     Website                1  
1                6         198.00        1188.00  Google Ads                1  
2                2         175.95         351.90  Google Ads                0  
3                1          86.25          86.25     Walk-in                1  
4                8         165.00        1320.00     Walk-in                0  
revie

In [4]:
import os
from openpyxl import Workbook

# Create folder structure
os.makedirs("data", exist_ok=True)

# Create workbook
wb = Workbook()
ws = wb.active
ws.title = "Targets"

# Add headers
ws.append(["KPI_Name", "Target_Value", "Unit", "Period"])

# Add rows manually
ws.append(["Occupancy Rate", 85, "%", "Monthly"])
ws.append(["Revenue per Unit", 120, "EUR", "Monthly"])
ws.append(["Avg Rental Duration", 8, "Months", "Yearly"])
ws.append(["Customer Satisfaction", 4.2, "/5.0", "Quarterly"])
ws.append(["Late Payment Rate", 5, "%", "Monthly"])

# Save file
wb.save("data/targets.xlsx")

print("targets.xlsx created successfully!")

targets.xlsx created successfully!


In [5]:
import pandas as pd
import re

# Read raw review file
with open("data/reviews.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

rows = []

for line in lines:
    line = line.strip()

    # Example format:
    # "Excellent service." - Helsinki, Jan 2024, 5/5

    match = re.match(r'"(.*)" - ([A-Za-z]+), ([A-Za-z]{3}) (\d{4}), (\d)/5', line)

    if match:
        comment = match.group(1)
        city = match.group(2)
        month = match.group(3)
        year = match.group(4)
        rating = int(match.group(5))

        rows.append({
            "city": city,
            "month": month,
            "year": year,
            "rating": rating,
            "comment": comment
        })

feedback = pd.DataFrame(rows)

feedback.to_csv("data/feedback_cleaned.csv", index=False)

print("feedback_cleaned.csv created successfully")
feedback.head()

feedback_cleaned.csv created successfully


,city,month,year,rating,comment
0,Espoo,Jun,2025,2,Price felt slightly high.
1,Vantaa,Jul,2025,5,Perfect solution during my move.
2,Tampere,Oct,2025,4,Perfect solution during my move.
3,Turku,Nov,2025,2,Customer service response was slow.
4,Helsinki,Jul,2025,5,Climate unit protected my furniture well.


In [8]:
import requests
import pandas as pd

cities = {
    "Helsinki": (60.17, 24.94),
    "Espoo": (60.21, 24.66),
    "Vantaa": (60.29, 25.04),
    "Tampere": (61.50, 23.76),
    "Turku": (60.45, 22.27)
}

all_data = []

for city, (lat, lon) in cities.items():

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2024-01-01",
        "end_date": "2025-12-31",
        "daily": "temperature_2m_mean",
        "timezone": "Europe/Helsinki"
    }

    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame({
        "date": data["daily"]["time"],
        "avg_temp": data["daily"]["temperature_2m_mean"]
    })

    df["date"] = pd.to_datetime(df["date"])
    df["city"] = city
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month

    monthly = df.groupby(["city", "year", "month"])["avg_temp"].mean().reset_index()

    all_data.append(monthly)

weather = pd.concat(all_data, ignore_index=True)

# Save in current folder first
weather.to_csv("data/weather_api.csv", index=False)

print("Final weather_api.csv created successfully")
weather.head(10)

Final weather_api.csv created successfully


,city,year,month,avg_temp
0,Helsinki,2024,1,-7.164516
1,Helsinki,2024,2,-3.510345
2,Helsinki,2024,3,-0.419355
3,Helsinki,2024,4,3.010000
4,Helsinki,2024,5,12.219355
5,Helsinki,2024,6,16.516667
6,Helsinki,2024,7,18.238710
7,Helsinki,2024,8,17.703226
8,Helsinki,2024,9,15.213333
9,Helsinki,2024,10,8.770968


In [11]:
import pandas as pd
import sqlite3

# Load files
bookings = pd.read_csv("data/bookings.csv")
weather = pd.read_csv("data/weather_api.csv")
feedback = pd.read_csv("data/feedback_cleaned.csv")

# Correct database location
conn = sqlite3.connect("database/nordicspace.db")

# Save tables
bookings.to_sql("bookings_raw", conn, if_exists="replace", index=False)
weather.to_sql("weather_raw", conn, if_exists="replace", index=False)
feedback.to_sql("feedback_raw", conn, if_exists="replace", index=False)

conn.close()

print("Database saved inside /database folder successfully.")

Database saved inside /database folder successfully.


In [12]:
import pandas as pd
import sqlite3

# Connect database
conn = sqlite3.connect("database/nordicspace.db")

# Load bookings
bookings = pd.read_sql("SELECT * FROM bookings_raw", conn)

# -------------------------
# DIM CITY
# -------------------------
dim_city = bookings[["city"]].drop_duplicates().reset_index(drop=True)
dim_city["city_id"] = dim_city.index + 1

# -------------------------
# DIM UNIT
# -------------------------
dim_unit = bookings[["unit_size", "monthly_price"]].drop_duplicates().reset_index(drop=True)
dim_unit["unit_id"] = dim_unit.index + 1

# -------------------------
# FACT BOOKINGS
# -------------------------
fact = bookings.merge(dim_city, on="city", how="left")
fact = fact.merge(dim_unit, on=["unit_size", "monthly_price"], how="left")

fact_bookings = fact[[
    "booking_id",
    "customer_id",
    "city_id",
    "unit_id",
    "duration_months",
    "monthly_price",
    "total_revenue",
    "repeat_customer"
]]

# Save tables
dim_city.to_sql("dim_city", conn, if_exists="replace", index=False)
dim_unit.to_sql("dim_unit", conn, if_exists="replace", index=False)
fact_bookings.to_sql("fact_bookings", conn, if_exists="replace", index=False)

conn.close()

print("Star schema created successfully.")
print("Tables:")
print("- dim_city")
print("- dim_unit")
print("- fact_bookings")

Star schema created successfully.
Tables:
- dim_city
- dim_unit
- fact_bookings


In [13]:
import shutil
import sqlite3

# Backup database first
shutil.copy("database/nordicspace.db", "database/nordicspace_backup.db")

# Export SQL separately
conn = sqlite3.connect("database/nordicspace.db")

with open("database/nordicspace.sql", "w", encoding="utf-8") as f:
    for line in conn.iterdump():
        f.write(line + "\n")

conn.close()

print("Backup created: nordicspace_backup.db")
print("SQL exported: nordicspace.sql")

Backup created: nordicspace_backup.db
SQL exported: nordicspace.sql
